# AURA - Hindi Speech-to-Text Fine-tuning
### Model: `openai/whisper-small` + LoRA (PEFT)
### Dataset: Google FLEURS Hindi (`hi_in`) — (https://huggingface.co/datasets/google/fleurs)
### Strategy: LoRA fine-tuning for efficient training on free T4 GPU

---
**Notebook Link:** *(https://colab.research.google.com/drive/1gVqnSR7ueJUB6LAw0JtDXx4KzN_C50Jd?usp=sharing)*  
**Video Link:** *(https://drive.google.com/file/d/17KD3VfuFM4-J6EGf-_qKE_fZhlLflI3u/view?usp=sharing)*

---



---

## Cell 1 — Environment Setup & Dependency Fix

In [ ]:
!pip install -q --upgrade pip
!pip install -q \
    transformers==4.41.0 \
    datasets==2.18.0 \
    accelerate==0.29.3 \
    peft==0.10.0 \
    evaluate==0.4.1 \
    jiwer==3.0.3 \
    librosa==0.10.1 \
    soundfile==0.12.1 \
    fsspec==2025.3.0 \
    tensorboard

print("All packages installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.2.0 which is incompatible.
All packages installed.


## Cell 2 — Imports & GPU Verification

In [ ]:
import os
import gc
import torch
import numpy as np
from dataclasses import dataclass
from typing import Any, Dict, List, Union

import evaluate
from datasets import load_dataset, Audio, DatasetDict
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model, PeftModel

# Verify GPU
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Cell 3 — Load Google FLEURS Hindi Dataset

**Why FLEURS instead of Common Voice 13:**
- Common Voice 13 is a gated dataset — requires accepting terms on Hugging Face and setting `HF_TOKEN` as a Colab secret
- FLEURS (`google/fleurs`, config `hi_in`) is fully public — no token, no login, no gating
- Same quality bar: FLEURS is Google's Few-shot Learning Evaluation of Universal Representations of Speech, covering 102 languages with professional-quality recordings
- Pre-split into train / validation / test — no data leakage risk
- Audio already at 16kHz — no resampling needed
- Text field: `transcription` (normalized, punctuation-cleaned Hindi text)

**Why 3000 training samples:**
- FLEURS Hindi train split has ~3700 samples total — 3000 gives strong coverage while leaving headroom
- At effective batch size 16 with 2000 steps, 3000 samples = ~10 passes through the data
- Training completes in ~2–2.5 hours on T4 — within the free session limit

In [ ]:
print("Loading Google FLEURS Hindi (hi_in)...")
print("No authentication required — fully public dataset.")

raw_dataset = load_dataset("google/fleurs", "hi_in", trust_remote_code=True)

print(f"\nRaw split sizes:")
for split in raw_dataset:
    print(f"  {split}: {len(raw_dataset[split])} samples")

print(f"\nColumns: {raw_dataset['train'].column_names}")

Loading Google FLEURS Hindi (hi_in)...
No authentication required — fully public dataset.


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]


Raw split sizes:
  train: 2120 samples
  validation: 239 samples
  test: 418 samples

Columns: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id']


In [ ]:
# ---------------------------------------------------------------
# Data Quality Filtering
# ---------------------------------------------------------------
# FLEURS is already curated — no upvote filtering needed.
# We only remove clips outside 1-15 seconds and empty transcriptions.
# FLEURS audio is already 16kHz so cast_column just confirms the format.

def is_valid_sample(batch):
    """Keep clips between 1 and 15 seconds with non-empty transcription."""
    audio    = batch["audio"]
    duration = len(audio["array"]) / audio["sampling_rate"]
    return (
        1.0 <= duration <= 15.0 and
        len(batch["transcription"].strip()) > 0
    )

print("Filtering train split...")
train_filtered = raw_dataset["train"].cast_column("audio", Audio(sampling_rate=16000))
train_filtered = train_filtered.filter(is_valid_sample, num_proc=1)
print(f"Train after filtering: {len(train_filtered)} samples")

print("Filtering validation split...")
val_filtered = raw_dataset["validation"].cast_column("audio", Audio(sampling_rate=16000))
val_filtered = val_filtered.filter(is_valid_sample, num_proc=1)
print(f"Validation after filtering: {len(val_filtered)} samples")

print("Casting test split (no filtering — preserve original distribution)...")
test_raw = raw_dataset["test"].cast_column("audio", Audio(sampling_rate=16000))
print(f"Test samples: {len(test_raw)} samples")

# Sample subset for training
TRAIN_SAMPLES = 3000
VAL_SAMPLES   = 500
TEST_SAMPLES  = 500

train_subset = train_filtered.shuffle(seed=42).select(range(min(TRAIN_SAMPLES, len(train_filtered))))
val_subset   = val_filtered.shuffle(seed=42).select(range(min(VAL_SAMPLES,   len(val_filtered))))
test_subset  = test_raw.shuffle(seed=42).select(range(min(TEST_SAMPLES,  len(test_raw))))

print(f"\nFinal sizes — Train: {len(train_subset)} | Val: {len(val_subset)} | Test: {len(test_subset)}")

Filtering train split...


Filter:   0%|          | 0/2120 [00:00<?, ? examples/s]

Train after filtering: 1799 samples
Filtering validation split...


Filter:   0%|          | 0/239 [00:00<?, ? examples/s]

Validation after filtering: 202 samples
Casting test split (no filtering — preserve original distribution)...
Test samples: 418 samples

Final sizes — Train: 1799 | Val: 202 | Test: 418


## Cell 4 — Load Whisper Processor (Feature Extractor + Tokenizer)

**Why `whisper-small`:**
- 244M parameters — trainable with LoRA on T4 (15GB VRAM)
- Native Hindi language token `<|hi|>` — no language head modification needed
- Encoder-decoder with cross-attention — handles noisy/accented speech better than CTC models
- Seq2SeqTrainer support built into Hugging Face transformers

In [ ]:
MODEL_ID = "openai/whisper-small"
LANGUAGE = "Hindi"
TASK     = "transcribe"

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)
processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)

print(f"Loaded processor for model: {MODEL_ID}")
print(f"Language token: {tokenizer.prefix_tokens}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loaded processor for model: openai/whisper-small
Language token: [50258, 50276, 50359, 50363]


## Cell 5 — Preprocessing: Audio → Log-Mel Spectrogram, Text → Token IDs

In [ ]:
def prepare_dataset(batch):
    """
    Convert raw audio and transcript into model-ready inputs.

    Audio path: raw waveform (16kHz) → 80-bin log-mel spectrogram
    Text path:  Hindi sentence → Whisper tokenizer → token ID list
    """
    audio = batch["audio"]

    # Compute log-mel spectrogram from waveform
    batch["input_features"] = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    # Encode transcript to token IDs
    # FLEURS uses "transcription" column (not "sentence" like Common Voice)
    batch["labels"] = tokenizer(batch["transcription"]).input_ids
    return batch


print("Preprocessing train split...")
train_dataset = train_subset.map(
    prepare_dataset,
    remove_columns=train_subset.column_names,
    num_proc=1
)

print("Preprocessing validation split...")
val_dataset = val_subset.map(
    prepare_dataset,
    remove_columns=val_subset.column_names,
    num_proc=1
)

print("Preprocessing test split...")
test_dataset = test_subset.map(
    prepare_dataset,
    remove_columns=test_subset.column_names,
    num_proc=1
)

print(f"\nDone. Sample input_features shape: {np.array(train_dataset[0]['input_features']).shape}")
print(f"Sample labels length: {len(train_dataset[0]['labels'])} tokens")

Preprocessing train split...


Map:   0%|          | 0/1799 [00:00<?, ? examples/s]

Preprocessing validation split...


Map:   0%|          | 0/202 [00:00<?, ? examples/s]

Preprocessing test split...


Map:   0%|          | 0/418 [00:00<?, ? examples/s]


Done. Sample input_features shape: (80, 3000)
Sample labels length: 80 tokens


## Cell 6 — Data Collator (Handles Variable-Length Batching)

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """
    Custom collator that pads audio spectrograms and token sequences
    to uniform length within each batch.

    Labels are padded with -100 so the loss function ignores
    padding positions during training.
    """
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Separate audio features and labels
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        # Pad spectrograms
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Pad token sequences, replace pad_token_id with -100
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Remove BOS token if it was prepended during tokenization
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("Data collator ready.")

Data collator ready.


## Cell 7 — WER & CER Metric Functions

In [ ]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    """
    Called automatically every eval_steps during training.
    Decodes predictions and references, then computes WER and CER.

    WER (Word Error Rate): fraction of words predicted incorrectly
    CER (Character Error Rate): fraction of characters predicted incorrectly
    Both metrics: lower is better.
    """
    pred_ids   = pred.predictions
    label_ids  = pred.label_ids

    # Replace -100 padding back to pad_token_id before decoding
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    cer = 100 * cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": round(wer, 2), "cer": round(cer, 2)}

print("Metrics loaded: WER + CER")

Metrics loaded: WER + CER


## Cell 8 — Load Whisper-Small & Apply LoRA

**Why LoRA instead of full fine-tuning:**
- Full fine-tuning updates all 244M parameters — very slow on T4, risks catastrophic forgetting
- LoRA injects small trainable rank-decomposition matrices into attention layers only
- We train ~1.5M parameters (0.6% of model) instead of 244M
- Training is 3-4x faster, uses 40% less GPU memory
- Comparable WER improvement to full fine-tuning at this data scale

In [ ]:
# Load base Whisper-small
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

# Configure generation for Hindi transcription
model.generation_config.language = "hindi"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None  # Let LoRA handle language tokens

# Enable gradient checkpointing to save VRAM
# Trades compute for memory: recomputes activations during backward pass
model.gradient_checkpointing_enable()

# Required when using gradient checkpointing with encoder-decoder models
model.config.use_cache = False

print(f"Base model loaded. Total parameters: {sum(p.numel() for p in model.parameters()):,}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

Base model loaded. Total parameters: 241,734,912


In [ ]:
# Apply LoRA configuration
# target_modules: apply adapters to query and value projection matrices
# in the transformer attention layers — these are the most impactful
# layers for language adaptation

lora_config = LoraConfig(
    r=32,                                        # Rank: higher = more capacity
    lora_alpha=64,                               # Scaling factor (alpha/r = 2.0)
    target_modules=["q_proj", "v_proj"],         # Attention query and value projections
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)

# Show trainable vs frozen parameter counts
trainable     = sum(p.numel() for p in model.parameters() if p.requires_grad)
total         = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} ({100 * trainable / total:.2f}% of total)")
print(f"Frozen parameters:    {total - trainable:,}")
print(f"Total parameters:     {total:,}")

Trainable parameters: 3,538,944 (1.44% of total)
Frozen parameters:    241,734,912
Total parameters:     245,273,856


## Cell 9 — Training Configuration & Launch

**Key hyperparameter decisions:**
- `lr=1e-4`: LoRA can tolerate higher LR than full fine-tuning (which needs `1e-5`)
- `warmup_steps=100`: Prevents loss spike in the critical first 100 steps
- `fp16=True`: T4 has Tensor Cores; FP16 gives ~30% speedup with no accuracy loss
- `gradient_accumulation_steps=2`: Simulates batch-16 without extra VRAM
- `max_steps=2000`: 1000 steps is under-trained for 3000 samples; 2000 converges better
- `eval_steps=200`: Checkpoint and evaluate every 200 steps to track progress

In [ ]:
OUTPUT_DIR = "./whisper-small-hindi-lora"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Batch & accumulation
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,          # Effective batch size = 16

    # Optimizer
    learning_rate=1e-4,
    warmup_steps=100,
    max_steps=2000,

    # Mixed precision
    fp16=True,

    # Evaluation & checkpointing
    evaluation_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,                # Lower WER = better

    # Logging
    logging_steps=25,
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="tensorboard",

    # Generation during evaluation
    predict_with_generate=True,
    generation_max_length=225,

    # Misc
    dataloader_num_workers=2,
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer configured.")
print(f"Steps per epoch: {len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")
print(f"Total training steps: {training_args.max_steps}")
print(f"Estimated time: ~2.0-2.5 hours on T4")

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:469: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


Trainer configured.
Steps per epoch: 112
Total training steps: 2000
Estimated time: ~2.0-2.5 hours on T4


In [ ]:
# ---------------------------------------------------------------
# START TRAINING
# Evaluation runs every 200 steps and prints WER + CER
# Best checkpoint is auto-saved based on lowest WER
# Expected total time: 2.0 - 2.5 hours on T4 GPU
# ---------------------------------------------------------------

print("Starting training...")
print("WER and CER will be printed every 200 steps.\n")

trainer.train()

print("\nTraining complete!")
print(f"Best model saved to: {OUTPUT_DIR}")

Starting training...
WER and CER will be printed every 200 steps.



/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss,Validation Loss,Wer,Cer
200,0.545400,0.568949,45.450000,17.820000
400,0.363900,0.405983,40.520000,15.770000
600,0.298200,0.375597,37.520000,14.660000
800,0.274900,0.364342,37.160000,15.550000


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_downloa

Step,Training Loss,Validation Loss,Wer,Cer
200,0.545400,0.568949,45.450000,17.820000
400,0.363900,0.405983,40.520000,15.770000
600,0.298200,0.375597,37.520000,14.660000
800,0.274900,0.364342,37.160000,15.550000
1000,0.251800,0.355897,35.320000,13.910000
1200,0.230400,0.356516,35.320000,13.820000
1400,0.214900,0.354429,34.400000,13.750000
1600,0.186400,0.354001,34.770000,14.070000
1800,0.197200,0.355667,34.650000,13.820000
2000,0.175600,0.356266,34.210000,14.030000


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_downloa


Training complete!
Best model saved to: ./whisper-small-hindi-lora


In [ ]:
from google.colab import drive
import os

# Mount Drive if not already mounted
drive.mount("/content/drive", force_remount=False)

# Save model + processor to Drive
save_path = "/content/drive/MyDrive/whisper-small-hindi-lora"
model.save_pretrained(save_path)
processor.save_pretrained(save_path)

# Confirm everything saved correctly
print("Files saved to Drive:")
for f in os.listdir(save_path):
    print(f"  {f}")

print(f"\nModel permanently saved at: {save_path}")
print("You can now safely close Colab — model will not be lost.")

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Files saved to Drive:
  README.md
  adapter_model.safetensors
  adapter_config.json
  preprocessor_config.json
  tokenizer_config.json
  special_tokens_map.json
  added_tokens.json
  vocab.json
  merges.txt
  normalizer.json

Model permanently saved at: /content/drive/MyDrive/whisper-small-hindi-lora
You can now safely close Colab — model will not be lost.


Just run these 3 cells to reload everything without retraining:


In [ ]:
# # Step 1 — Mount Drive
# from google.colab import drive
# drive.mount("/content/drive")

# # Step 2 — Reinstall packages (always needed after restart)
# # Run Cell 1 (pip install)

# # Step 3 — Reload imports, processor, dataset
# # Run Cells 2, 3, 4 normally

# # Step 4 — Load saved model directly from Drive (skip training entirely)
# from peft import PeftModel
# from transformers import WhisperForConditionalGeneration

# MODEL_ID   = "openai/whisper-small"
# OUTPUT_DIR = "/content/drive/MyDrive/whisper-small-hindi-lora"

# base  = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
# model = PeftModel.from_pretrained(base, OUTPUT_DIR, local_files_only=True)
# model = model.to("cuda")
# model.eval()

# print("Fine-tuned model loaded from Drive. Ready to use.")

In [ ]:
# Save the final LoRA adapter weights and processor
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Model and processor saved to {OUTPUT_DIR}")

Model and processor saved to ./whisper-small-hindi-lora


## Cell 10 — Evaluate Base Whisper-Small on Test Set

We evaluate the **unmodified** base model first to establish the baseline WER and CER.  
This is run on the same 500 test samples we will use for the fine-tuned model.

In [ ]:
from tqdm import tqdm

def evaluate_model_on_test(model_or_id, test_data, processor, label="Model", is_peft=False, batch_size=16):
    """
    Run inference on test_data and compute WER + CER.

    Parameters
    ----------
    model_or_id : str or model object
        Either a model ID string (loads fresh) or an already-loaded model.
    test_data   : HuggingFace Dataset with 'audio' and 'transcription' columns
    processor   : WhisperProcessor
    label       : str, display name for printout
    is_peft     : bool, whether this is a LoRA PEFT model
    batch_size  : int
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"

    if isinstance(model_or_id, str):
        if is_peft:
            base = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
            eval_model = PeftModel.from_pretrained(base, model_or_id)
        else:
            eval_model = WhisperForConditionalGeneration.from_pretrained(model_or_id)
    else:
        eval_model = model_or_id

    eval_model = eval_model.to(device)
    eval_model.eval()

    all_predictions = []
    all_references  = []

    forced_decoder_ids = processor.get_decoder_prompt_ids(language="hindi", task="transcribe")

    for i in tqdm(range(0, len(test_data), batch_size), desc=f"Evaluating {label}"):
        batch = test_data[i : i + batch_size]

        # Process each clip individually to guarantee fixed 3000-frame padding.
        # Batched processor call only pads to the longest clip in the batch,
        # which can be shorter than Whisper's required 3000 frames.
        input_features = torch.stack([
            torch.tensor(
                processor.feature_extractor(
                    a["array"], sampling_rate=16000
                ).input_features[0]
            )
            for a in batch["audio"]
        ]).to(device)

        with torch.no_grad():
            predicted_ids = eval_model.generate(
                input_features,
                forced_decoder_ids=forced_decoder_ids,
                max_new_tokens=225
            )

        predictions = processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)
        all_predictions.extend(predictions)
        all_references.extend(batch["transcription"])

    wer = 100 * wer_metric.compute(predictions=all_predictions, references=all_references)
    cer = 100 * cer_metric.compute(predictions=all_predictions, references=all_references)

    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"  Samples evaluated: {len(all_predictions)}")
    print(f"  WER: {wer:.2f}%")
    print(f"  CER: {cer:.2f}%")
    print(f"{'='*50}\n")

    # Clean up GPU memory
    eval_model.cpu()
    del eval_model
    gc.collect()
    torch.cuda.empty_cache()

    return wer, cer, all_predictions, all_references


print("Evaluation function ready.")

Evaluation function ready.


In [ ]:
# Evaluate BASE model (no fine-tuning)
print("Step 1/2 — Evaluating BASE Whisper-small on test set...")

base_wer, base_cer, base_preds, references = evaluate_model_on_test(
    model_or_id=MODEL_ID,
    test_data=test_subset,
    processor=processor,
    label="Base Whisper-small (no fine-tuning)",
    is_peft=False
)

Step 1/2 — Evaluating BASE Whisper-small on test set...


Evaluating Base Whisper-small (no fine-tuning): 100%|██████████| 27/27 [05:12<00:00, 11.58s/it]



  Base Whisper-small (no fine-tuning)
  Samples evaluated: 418
  WER: 68.86%
  CER: 34.62%



## Cell 11 — Evaluate Fine-Tuned LoRA Model on Same Test Set

In [ ]:
# Evaluate FINE-TUNED LoRA model
print("Step 2/2 — Evaluating FINE-TUNED Whisper-small (LoRA) on test set...")

finetuned_wer, finetuned_cer, finetuned_preds, _ = evaluate_model_on_test(
    model_or_id=OUTPUT_DIR,
    test_data=test_subset,
    processor=processor,
    label="Fine-tuned Whisper-small + LoRA (Hindi)",
    is_peft=True
)

Step 2/2 — Evaluating FINE-TUNED Whisper-small (LoRA) on test set...


Evaluating Fine-tuned Whisper-small + LoRA (Hindi): 100%|██████████| 27/27 [04:55<00:00, 10.93s/it]



  Fine-tuned Whisper-small + LoRA (Hindi)
  Samples evaluated: 418
  WER: 34.92%
  CER: 14.33%



## Cell 12 — Results Comparison Table

In [ ]:
wer_improvement = base_wer - finetuned_wer
cer_improvement = base_cer - finetuned_cer
wer_pct_drop    = (wer_improvement / base_wer) * 100
cer_pct_drop    = (cer_improvement / base_cer) * 100

print("\n" + "="*62)
print("   EVALUATION RESULTS — Base vs Fine-tuned Whisper-small")
print("="*62)
print(f"  {'Metric':<10} {'Base Model':>14} {'Fine-tuned':>14} {'Improvement':>14}")
print("-"*62)
print(f"  {'WER':<10} {base_wer:>13.2f}% {finetuned_wer:>13.2f}% {wer_improvement:>+13.2f}%")
print(f"  {'CER':<10} {base_cer:>13.2f}% {finetuned_cer:>13.2f}% {cer_improvement:>+13.2f}%")
print("="*62)
print(f"\n  WER reduced by {wer_pct_drop:.1f}%  |  CER reduced by {cer_pct_drop:.1f}%")
print(f"  Test samples: {len(references)} (held-out, never seen during training)")
print(f"  Dataset: Google FLEURS Hindi (hi_in)")
print(f"  Training samples: {TRAIN_SAMPLES} (filtered from {len(train_filtered)} available)")
print("="*62)


   EVALUATION RESULTS — Base vs Fine-tuned Whisper-small
  Metric         Base Model     Fine-tuned    Improvement
--------------------------------------------------------------
  WER                68.86%         34.92%        +33.94%
  CER                34.62%         14.33%        +20.29%

  WER reduced by 49.3%  |  CER reduced by 58.6%
  Test samples: 418 (held-out, never seen during training)
  Dataset: Google FLEURS Hindi (hi_in)
  Training samples: 3000 (filtered from 1799 available)


## Cell 13 — Example Transcriptions (Qualitative Analysis)

In [ ]:
print("EXAMPLE TRANSCRIPTIONS (first 10 test samples)")
print("="*80)

for i in range(min(10, len(references))):
    ref  = references[i]
    base = base_preds[i]
    fine = finetuned_preds[i]

    print(f"\nSample {i+1}:")
    print(f"  Reference   : {ref}")
    print(f"  Base model  : {base}")
    print(f"  Fine-tuned  : {fine}")
    print("-"*80)

EXAMPLE TRANSCRIPTIONS (first 10 test samples)

Sample 1:
  Reference   : ऐसा माना जाता है कि पेरिस के निवासी अहंकारी असभ्य और अभिमानी होते हैं
  Base model  :  आईसा माना जाता है कि पैरिस की निवासी एंकारी असवब़ और अबिमानी होते है
  Fine-tuned  : ऐसा माना जाता है कि पैरिस की निवासी अयंकारी असब्ब्ब और अबिमानी होते हैं
--------------------------------------------------------------------------------

Sample 2:
  Reference   : कुछ अणुओं में अस्थिर केंद्रक होता है जिसका मतलब यह है कि उनमें थोड़े या बिना किसी झटके से टूटने की प्रवृत्ति होती है
  Base model  :  अग्वो में आज्द्टर केंद्रख होता है, जिसका मतला भी आजा की उन्मे थोडे या बिना किसी जटके से तुटनें की प्रवत्ती होती है.
  Fine-tuned  : कुछ अड़ुमों में आसत्र केंद्रक होता है जिसका मतलां वियाहा की उनमें थोड़े या बिना किछी झटके से ठूटने की प्रवत्ती होती है
--------------------------------------------------------------------------------

Sample 3:
  Reference   : हालांकि शेंगेन ज़ोन इस मामले में कुछ हद तक एक देश की तरह काम करता है
  Base model

## Cell 14 — Live Demo with Random Test Sample

**This is the cell for demo screen-recording**

It:
1. Picks a random audio clip from the test set
2. Plays the audio in the notebook
3. Runs it through the fine-tuned model
4. Prints the reference transcript and model prediction side by side

In [ ]:
import random
import IPython.display as ipd

# Pick a random clip
idx = random.randint(0, len(test_subset) - 1)
sample = test_subset[idx]

audio_array   = sample["audio"]["array"]
sampling_rate = sample["audio"]["sampling_rate"]
reference     = sample["transcription"]   # FLEURS uses "transcription" column

print(f"Demo Sample Index: {idx}")
print(f"Audio duration: {len(audio_array) / sampling_rate:.1f} seconds")
print(f"\nReference transcript: {reference}")
print("\nPlaying audio...")

# Play audio in notebook
display(ipd.Audio(audio_array, rate=sampling_rate))

Demo Sample Index: 306
Audio duration: 8.5 seconds

Reference transcript: वर्तमान में जो कीड़े अपने पंखों को वापस नहीं मोड़ सकते हैं उनमें केवल ड्रैगनफ़्लाई और मेफलीज़ हैं

Playing audio...


In [ ]:
# Load fine-tuned model for the demo
device = "cuda" if torch.cuda.is_available() else "cpu"

base_for_demo   = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
demo_model      = PeftModel.from_pretrained(base_for_demo, OUTPUT_DIR)
demo_model      = demo_model.to(device)
demo_model.eval()

forced_decoder_ids = processor.get_decoder_prompt_ids(language="hindi", task="transcribe")

# Prepare input
input_features = processor(
    audio_array,
    sampling_rate=sampling_rate,
    return_tensors="pt"
).input_features.to(device)

# Run base model
base_model_demo = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
base_model_demo.eval()
with torch.no_grad():
    base_ids = base_model_demo.generate(input_features, forced_decoder_ids=forced_decoder_ids, max_new_tokens=225)
base_output = processor.tokenizer.decode(base_ids[0], skip_special_tokens=True)

# Run fine-tuned model
with torch.no_grad():
    fine_ids = demo_model.generate(input_features, forced_decoder_ids=forced_decoder_ids, max_new_tokens=225)
fine_output = processor.tokenizer.decode(fine_ids[0], skip_special_tokens=True)

# Print side-by-side comparison
print("\n" + "="*70)
print("  DEMO — Single Audio Clip Transcription")
print("="*70)
print(f"  Reference (ground truth) : {reference}")
print(f"  Base Whisper-small       : {base_output}")
print(f"  Fine-tuned (LoRA Hindi)  : {fine_output}")
print("="*70)


  DEMO — Single Audio Clip Transcription
  Reference (ground truth) : वर्तमान में जो कीड़े अपने पंखों को वापस नहीं मोड़ सकते हैं उनमें केवल ड्रैगनफ़्लाई और मेफलीज़ हैं
  Base Whisper-small       :  अगर तमान में जो कीडे अपने पंको को वापस नहीं मोर सकते हैं उन में केबल ट्रागन फ्लाय और मैफलीज हैं
  Fine-tuned (LoRA Hindi)  : अर तमान में जो कीडे अपने पंखों को वापस नहीं मौर सकते हैं उनमें केबल ड्रागनफ्लाय और मैफलीज़ हैं


In [ ]:
import random
import IPython.display as ipd

# ── Load fine-tuned model once ───────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"

base_for_demo  = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
base_for_demo.eval()

ft_base        = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
ft_model       = PeftModel.from_pretrained(ft_base, OUTPUT_DIR).to(device)
ft_model.eval()

forced_decoder_ids = processor.get_decoder_prompt_ids(language="hindi", task="transcribe")

def transcribe(model, audio_array):
    feats = processor.feature_extractor(
        audio_array, sampling_rate=16000, return_tensors="pt"
    ).input_features.to(device)
    with torch.no_grad():
        ids = model.generate(feats, forced_decoder_ids=forced_decoder_ids, max_new_tokens=225)
    return processor.tokenizer.decode(ids[0], skip_special_tokens=True)

# ── Pick 3 random clips from test set ────────────────────────────────
random.seed(99)
indices = random.sample(range(len(test_subset)), 3)

for clip_num, idx in enumerate(indices, start=1):
    sample        = test_subset[idx]
    audio_array   = sample["audio"]["array"]
    sampling_rate = sample["audio"]["sampling_rate"]
    reference     = sample["transcription"]
    duration      = len(audio_array) / sampling_rate

    base_out = transcribe(base_for_demo, audio_array)
    fine_out = transcribe(ft_model,      audio_array)

    print(f"\n{'='*70}")
    print(f"  CLIP {clip_num} of 3  |  Index: {idx}  |  Duration: {duration:.1f}s")
    print(f"{'='*70}")
    print(f"  Reference (ground truth) : {reference}")
    print(f"  Base Whisper-small       : {base_out}")
    print(f"  Fine-tuned (LoRA Hindi)  : {fine_out}")
    print(f"{'='*70}")

    print(f"\n  Playing Clip {clip_num}...")
    display(ipd.Audio(audio_array, rate=sampling_rate))
    print()

# ── Cleanup ───────────────────────────────────────────────────────────
base_for_demo.cpu()
ft_model.cpu()
del base_for_demo, ft_model
gc.collect()
torch.cuda.empty_cache()
print("\nDemo complete.")


  CLIP 1 of 3  |  Index: 206  |  Duration: 9.0s
  Reference (ground truth) : 11:35 बजे अग्नि बचाव दल ने अंततः आग बुझा दी थी
  Base Whisper-small       :  इग्यारा प्झिज़ बजे अगनी बचाओ दल ने अंतता आग भुजा दी थी
  Fine-tuned (LoRA Hindi)  : 11.35 बजे अगनी बचाव दल ने अंतता आंग भूजा दी थी

  Playing Clip 1...




  CLIP 2 of 3  |  Index: 194  |  Duration: 8.4s
  Reference (ground truth) : यह कहते हुए कि वे चीन के आर्थिक उत्पादन के आधार पर बनाए जाएंगे उन्होंने कटौती के लिए कोई आंकड़ा निर्धारित नहीं किया
  Base Whisper-small       :  यह कहते हुयो कि वे चीन के आर्थे कुटबादन के आदर पर बनाई जाएंगे उना निक तरोती कि लिये कुई आंक्डा निर्दारित नहीं किया
  Fine-tuned (LoRA Hindi)  : यह कहते हुए कि वे चीन के आर्थिक उत्पादन के आदर पर बनाए जाएंगे उन्होंनी कट्रोथी के लिए को यांखड़ा निर्दारित नहीं किया

  Playing Clip 2...




  CLIP 3 of 3  |  Index: 102  |  Duration: 6.7s
  Reference (ground truth) : इसलिए यह माना जा रहा है कि नोटेशन को सिर्फ़ एक लेबल के रूप में जोड़ा गया था
  Base Whisper-small       :  इस्टिए ये माना जा रहा है की, नोटेशन को सिव एक लेबल की रूप में जोडा गया था.
  Fine-tuned (LoRA Hindi)  : इसलि यह माना जा रहा है कि नोटेशन को सिव एक लेबल की रूप में जोड़ा गया था

  Playing Clip 3...




Demo complete.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
